In [2]:
import sys
print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [3]:
import pandas as pd


In [14]:
csv_path = "../data/raw/sim_events_20251117_152933.csv"

In [15]:
csv_path =df = pd.read_csv(csv_path)
df.head()

,event_type,convoy_id,minute,step_index,progress
0,convoy_spawned,1,0,-1,0.0
1,convoy_spawned,2,0,-1,0.0
2,convoy_spawned,3,0,-1,0.0
3,convoy_spawned,4,0,-1,0.0
4,convoy_spawned,5,0,-1,0.0


In [16]:
len(df)

5

In [17]:
df.columns.tolist()

['event_type', 'convoy_id', 'minute', 'step_index', 'progress']

In [18]:
df.dtypes

event_type     object
convoy_id       int64
minute          int64
step_index      int64
progress      float64
dtype: object

In [19]:
df["event_type"].value_counts()

event_type
convoy_spawned    5
Name: count, dtype: int64

In [20]:
n_convoys = df["convoy_id"].nunique()
n_convoys

5

In [21]:
df_completed = df[df["event_type"] == "convoy_completed"]

n_completed_events = len(df_completed)
n_completed_events

0

In [22]:
n_completed_convoys = df_completed["convoy_id"].nunique()
n_completed_convoys


0

In [23]:
n_not_completed_convoys = n_convoys - n_completed_convoys
n_not_completed_convoys

5

In [24]:
completion_minutes_by_convoy = (
    df_completed
    .groupby("convoy_id")["minute"]
    .min()
)

completion_minutes_by_convoy

Series([], Name: minute, dtype: int64)

In [25]:
avg_completion_minute = completion_minutes_by_convoy.mean()
max_completion_minute = completion_minutes_by_convoy.max()
min_completion_minute = completion_minutes_by_convoy.min()

avg_completion_minute, min_completion_minute, max_completion_minute

(nan, nan, nan)

In [26]:
kpis = {
    "n_convoys": int(n_convoys),
    "n_completed_convoys": int(n_completed_convoys),
    "n_not_completed_convoys": int(n_not_completed_convoys),
    "avg_completion_minute": float(avg_completion_minute) if n_completed_convoys > 0 else None,
    "min_completion_minute": float(min_completion_minute) if n_completed_convoys > 0 else None,
    "max_completion_minute": float(max_completion_minute) if n_completed_convoys > 0 else None,
}

kpis

{'n_convoys': 5,
 'n_completed_convoys': 0,
 'n_not_completed_convoys': 5,
 'avg_completion_minute': None,
 'min_completion_minute': None,
 'max_completion_minute': None}

In [29]:
import matplotlib.pyplot as plt

In [30]:
if len(completion_minutes_by_convoy) > 0:
    plt.figure()
    plt.hist(completion_minutes_by_convoy, bins=10)
    plt.xlabel("Minuto di completamento")
    plt.ylabel("Numero di convogli")
    plt.title("Distribuzione dei tempi di completamento dei convogli")
else:
    print("Nessun convoglio completato: impossibile creare l'istogramma.")

Nessun convoglio completato: impossibile creare l'istogramma.


In [31]:
if len(df_completed) > 0:
    # Conta quanti completamenti avvengono a ogni minuto
    completed_per_minute = (
        df_completed
        .groupby("minute")["convoy_id"]
        .nunique()
        .sort_index()
    )

    # Calcola il cumulativo
    cumulative_completed = completed_per_minute.cumsum()

    plt.figure()
    plt.step(cumulative_completed.index, cumulative_completed.values, where="post")
    plt.xlabel("Minuto")
    plt.ylabel("Convogli completati (cumulativo)")
    plt.title("Convogli completati nel tempo (cumulativo)")
else:
    print("Nessun convoglio completato: impossibile creare la curva cumulativa.")

Nessun convoglio completato: impossibile creare la curva cumulativa.
